# Geometric defects of midpoint projection — section 4.9

Measure the complete physical one-step maps of ABBA2Midpoint and BM4Midpoint, with ABBA2Implicit and BM4Implicit as symmetric-projection controls. The reflected potential, normalized gyro-radius 0.015 and 84 particles are taken from the selected reference campaign. Four forcing phases and 20, 40, 80 and 160 fixed steps per cycle define the common-point sweep. At 40 steps per cycle, accumulate geometric diagnostics over 10 and 100 cycles.

The diagnostics are signed `det(D Phi)-1`, the correctly timed forward/backward displacement, copy separation, `det(D e)` and the residual of the determinant identity. Analytic field Jacobians propagate derivatives through the actual composition. A smooth oscillator calibration and a projection-tolerance sweep expose numerical floors. Long-time return displacements also contain perturbation amplification. Physical energy variation does not measure either geometric property.

**The default is to reuse saved data.** A new calculation requires `RUN_COMPUTATION=True` and a fresh run identifier. The final scientific destination is the B2 prefix below; the local directory is a recovery cache. See [the study protocol](../../docs/studies/midpoint-geometry.md).

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "src/studies/midpoint_geometry.py").is_file())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

STUDY_PREFIX = "studies/midpoint_geometry"
RUN_ID = "midpoint-geometry-20261118-rho0015"
RUN_DIRECTORY = ROOT / "outputs/midpoint_geometry" / RUN_ID
BUCKET = f"gc2d_data:gc2d-notebooks-data/{STUDY_PREFIX}/{RUN_ID}"
FILES = ("local.npz", "seams.npz", "fd_audit.npz", "calibration.npz", "implicit_controls.npz",
         "accumulation.npz", "summary.json", "manifest.json", "source.tar.gz")

RUN_COMPUTATION = False
RECOVER_FROM_BUCKET = True
ARCHIVE_RESULTS = False


## Load a completed record or explicitly compute a new one

The full calculation is intentionally outside routine notebook validation. A new run writes all six numerical archives, the summary and the manifest. Computation uses local NumPy float64; total diagnostic timing includes local archive writes and excludes network transfer and plotting. Recovery downloads saved files without integrating trajectories.

In [ ]:
if RUN_COMPUTATION:
    if any((RUN_DIRECTORY / name).exists() for name in FILES):
        raise FileExistsError("Use a new RUN_ID for a new calculation; retain completed or recoverable data.")
    from studies.midpoint_geometry import run_study
    summary = run_study(ROOT, RUN_DIRECTORY, accumulation_cycles=(10, 100))
else:
    missing = [name for name in FILES if not (RUN_DIRECTORY / name).is_file()]
    if missing and RECOVER_FROM_BUCKET:
        from hashlib import sha256
        from tempfile import TemporaryDirectory
        from diagnostics.io.storage import ArtifactStore
        store = ArtifactStore(BUCKET)
        with TemporaryDirectory(prefix="gc2d-midpoint-recovery-") as temporary:
            staging = Path(temporary)
            store.get("archive-manifest.json", staging / "archive-manifest.json")
            archive = json.loads((staging / "archive-manifest.json").read_text())
            if archive["bucket"] != BUCKET:
                raise ValueError("The archive manifest names a different study destination.")
            for name in FILES:
                store.get(name, staging / name)
                actual = sha256((staging / name).read_bytes()).hexdigest()
                if actual != archive["files"][name]["sha256"]:
                    raise ValueError(f"Downloaded file does not match its archived fingerprint: {name}")
                existing = RUN_DIRECTORY / name
                if existing.exists() and sha256(existing.read_bytes()).hexdigest() != actual:
                    raise ValueError(f"Local and bucket contents differ: {name}. Preserve both before recovery.")
            RUN_DIRECTORY.mkdir(parents=True, exist_ok=True)
            for name in missing:
                (RUN_DIRECTORY / name).write_bytes((staging / name).read_bytes())
    elif missing:
        raise FileNotFoundError(f"Missing completed study files: {missing}. Enable recovery or explicitly request a new run.")
    summary = json.loads((RUN_DIRECTORY / "summary.json").read_text())

if summary["run_id"] != RUN_ID or summary["status"] != "complete":
    raise ValueError("The saved record is incomplete or has a different run identifier.")
print(json.dumps(summary, indent=2))


In [ ]:
with np.load(RUN_DIRECTORY / "local.npz", allow_pickle=False) as local:
    assert local["det_defect"].shape == (2, 4, 4, 84)
    assert local["return_norm"].shape == local["det_defect"].shape
    assert local["identity_residual"].shape == local["det_defect"].shape
    print("Local diagnostic axes: method, step size, forcing phase, particle.")
    print("Steps per cycle:", local["steps_per_cycle"].tolist())
    print("Forcing phases:", local["phases"].tolist())
    print("Maximum determinant-identity residual:", float(np.nanmax(np.abs(local["identity_residual"]))))


## Publish and verify scientific results

Set `ARCHIVE_RESULTS=True` to transfer the completed record to the explicit B2 destination. Immutable uploads refuse different remote content. Each transferred file is downloaded again and checked byte for byte; failure leaves the local record intact and must not be reported as successful publication. Transfer time is excluded from numerical timings. Rerunning this cell never integrates trajectories.

In [ ]:
if ARCHIVE_RESULTS:
    import subprocess
    archive = subprocess.run(
        [sys.executable, str(ROOT / "scripts/archive_midpoint_geometry.py"), str(RUN_DIRECTORY)],
        cwd=ROOT, check=True, text=True,
    )
    publication = json.loads((RUN_DIRECTORY / "publication.json").read_text())
    assert publication["status"] == "verified" and publication["bucket"] == BUCKET
    print(json.dumps(publication, indent=2))
else:
    print(f"Publication not run by this notebook. Scientific destination: {BUCKET}")
